In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
from scipy import interpolate
from typing import Tuple, Optional
import os
import requests
from pathlib import Path

def download_ice6g_data(url, local_filename='ice6g_data.nc'):
    """Download ICE6G data reliably using requests."""
    
    # Check if file already exists
    if os.path.exists(local_filename):
        print(f"File {local_filename} already exists, loading...")
        return local_filename)
    
    # Download the file
    print(f"Downloading from {url}...")
    response = requests.get(url)
    
    if response.status_code == 200:
        print(f"Download successful! ({len(response.content)} bytes)")
        
        # Save to file
        with open(local_filename, 'wb') as f:
            f.write(response.content)
        
        return Path(local_filename).resolve()
    else:
        raise Exception(f"Download failed: {response.status_code}")


def load_gia_model(model_name: str, aoi: Optional[gpd.GeoDataFrame] = None, 
                   caron_path: Optional[str] = None, 
                   ice6g_path: Optional[str] = None) -> Tuple[callable, callable]:
    """
    Load and interpolate GIA model data (either Caron et al. 2018 or ICE6G-D).
    
    Parameters:
    -----------
    model_name : str
        Either 'caron' or 'ice6g' to specify which model to load
    aoi : gpd.GeoDataFrame, optional
        Area of interest for clipping the data. If None, uses global data.
    caron_path : str, optional
        Path to Caron et al. 2018 data file
    ice6g_path : str, optional
        Path to ICE6G-D NetCDF file
        
    Returns:
    --------
    Tuple[callable, callable]
        Returns (vlm_rate_interpolator, vlm_std_interpolator) for Caron model
        Returns (vlm_rate_interpolator, None) for ICE6G-D model
        
    Examples:
    ---------
    # Load Caron model with area of interest
    vlm_interp, std_interp = load_gia_model('caron', aoi, 
                                           caron_path='path/to/caron/file')
    
    # Load Caron model globally (no clipping)
    vlm_interp, std_interp = load_gia_model('caron', 
                                           caron_path='path/to/caron/file')
    
    # Load ICE6G-D model  
    vlm_interp, _ = load_gia_model('ice6g', aoi, 
                                  ice6g_path='path/to/ice6g.nc')
    """
    
    if model_name.lower() == 'caron':
        return _load_caron_model(caron_path, aoi)
    elif model_name.lower() == 'ice6g':
        return _load_ice6g_model(ice6g_path, aoi)
    else:
        raise ValueError(f"Unknown model_name: {model_name}. Use 'caron' or 'ice6g'")


def _load_caron_model(file_path: str, aoi: Optional[gpd.GeoDataFrame] = None) -> Tuple[callable, callable]:
    """Load and process Caron et al. 2018 GIA model."""
    if file_path is None:
        raise ValueError("caron_path must be provided when loading Caron model")
    
    # Define column names
    names = ['lat', 'lon', 'vlm_rate', 'vlm_std', 
             'geoid_rate', 'geoid_std', 'gravity_rate', 'gravity_std']
    
    # Read the data
    gia_df = pd.read_csv(file_path, skiprows=6, delimiter=r"\s+", 
                        names=names, dtype=float)
    
    # Shift longitude from 0-360 to -180-180
    gia_df.loc[gia_df.lon > 180, 'lon'] = gia_df.loc[gia_df.lon > 180, 'lon'] - 360
    
    # Adjust latitude coordinates
    gia_df.lat -= 90
    gia_df.lat = np.flipud(gia_df.lat)
    
    # Create GeoDataFrame
    caron_gdf = gpd.GeoDataFrame(gia_df,
                               geometry=gpd.points_from_xy(gia_df.lon, gia_df.lat),
                               crs='EPSG:4326')
    
    # Clip to area of interest only if aoi is provided
    if aoi is not None:
        caron_gdf = caron_gdf.clip(aoi)
    
    # Create interpolation splines
    vlm_spline = interpolate.SmoothBivariateSpline(caron_gdf.lon, 
                                                  caron_gdf.lat, 
                                                  caron_gdf.vlm_rate, 
                                                  kx=2, ky=2)
    
    std_spline = interpolate.SmoothBivariateSpline(caron_gdf.lon, 
                                                  caron_gdf.lat, 
                                                  caron_gdf.vlm_std, 
                                                  kx=2, ky=2)
    
    return vlm_spline, std_spline


def _load_ice6g_model(file_path: str, aoi: Optional[gpd.GeoDataFrame] = None) -> Tuple[callable, None]:
    """Load and process ICE6G-D GIA model."""
    if file_path is None:
        raise ValueError("ice6g_path must be provided when loading ICE6G-D model")
    
    # Read NetCDF data
    da_ice6g = xr.open_dataset(file_path)
    ice6_df = da_ice6g.to_dataframe().reset_index()
    
    # Shift longitude from 0-360 to -180-180
    ice6_df.loc[ice6_df.Lon > 180, 'Lon'] = ice6_df.loc[ice6_df.Lon > 180, 'Lon'] - 360
    
    # Create GeoDataFrame
    ice6_gdf = gpd.GeoDataFrame(ice6_df,
                              geometry=gpd.points_from_xy(ice6_df.Lon, ice6_df.Lat),
                              crs='EPSG:4326')
    
    # Rename columns for consistency
    ice6_gdf = ice6_gdf.rename(columns={'Drad_250': 'vlm_rate', 
                                       'Lon': 'lon', 
                                       'Lat': 'lat'})
    
    # Clip to area of interest only if aoi is provided
    if aoi is not None:
        ice6_gdf = ice6_gdf.clip(aoi)
    
    # Create interpolation spline
    coords = ice6_gdf.get_coordinates()
    vlm_spline = interpolate.SmoothBivariateSpline(coords.x.values, 
                                                  coords.y.values, 
                                                  ice6_gdf.vlm_rate, 
                                                  kx=2, ky=2)
    
    return vlm_spline, None

In [1]:
import opera_utils 
import venti
from venti.models import load_gia

## Load DISP frame

In [2]:
itrf=2014
frame_id=3060

disp_frame_db = opera_utils.get_frame_geodataframe()
selected_fid = disp_frame_db[disp_frame_db.index==frame_id]
selected_fid

,is_land,is_north_america,orbit_pass,geometry
frame_id,,,,
3060,1,True,DESCENDING,"POLYGON ((-102.03708 37.27968, -101.7256 38.81..."


In [3]:
# Caron et al 2018
gia_caron = load_gia.load_caron_model(load_gia.CARON_GIA)
# Clip the data around selected DISP frame
gia_caron_frame = load_gia.clip_gia_df(gia_caron, aoi=selected_fid)

In [6]:
# Peltier et al 2018
import shutil
from pathlib import Path
ice6_file = Path(load_gia.download_ice6g_data())
gia_ice6 = load_gia.load_ice6g_model(ice6_file)
# Delete temp file
ice6_file.unlink()

# Clip the data around selected DISP frame
gia_ice6_frame = load_gia.clip_gia_df(gia_ice6, aoi=selected_fid)

Download successful! (6492576 bytes)


In [7]:
import folium
from folium import LayerControl

# First map
m = gia_caron_frame.explore(column="vlm_rate", name="Caron GIA", cmap='RdBu')

# Add another dataset to the same map
gia_ice6_frame.explore(m=m, column="vlm_rate", name="ICE6D GIA")

selected_fid.explore(m=m, name='DISP frame')

# Add layer control so you can toggle layers
folium.LayerControl().add_to(m)

m

In [8]:
# Rasterize and export 
isce6d_raster, isce6d_attr = load_gia.rasterize_gdf(gia_ice6_frame, 'vlm_rate')
caron_raster, caron_attr = load_gia.rasterize_gdf(gia_caron_frame, 'vlm_rate')

# Export and project to OPERA DISP frame

In [11]:
import rasterio
import numpy as np
from rasterio.crs import CRS
from rasterio.transform import Affine, from_bounds
from rasterio.warp import Resampling, reproject

In [12]:
# Determine UTM CRS for the frame
utm_crs = selected_fid.estimate_utm_crs()
selected_frame_utm = selected_fid.to_crs(utm_crs)
minx, miny, maxx, maxy = selected_frame_utm.total_bounds
xs = np.arange(minx, maxx, 30)
ys = np.arange(maxy, miny, -30)
dst_transform = Affine(30, 0, minx, 0, -30, maxy)
target_height, target_width = len(ys), len(xs)
atr = {"rows":target_height, "cols":target_width}

In [13]:
output_name = f"frame_{frame_id}_GIA_Caron_rates.tif"
resampling_mode: Resampling = Resampling.bilinear


with rasterio.open(
        output_name, "w",
        height=atr["rows"], width=atr["cols"],
        count=1, dtype="float32", crs=utm_crs, transform=dst_transform,
        nodata=np.nan,
        compress='lzw', tiled=True, blockxsize=512, blockysize=512,
        predictor=2, interleave='band',
) as dst:
    dst.set_band_description(1, "GIA VLM (mm/yr)")

    reproject(
        source=caron_raster,
        destination=rasterio.band(dst, 1),
        src_transform=caron_attr['transform'],
        src_crs=caron_attr['crs'],
        dst_transform=dst_transform,
        dst_crs=utm_crs,
        resampling=resampling_mode,
        dst_nodata=np.nan,
    )